# Topic 12 - Structured Output

Build a parse-and-retry loop by hand. No Instructor, no SDK structured outputs — just Pydantic validation and retry logic that works on any provider.

## Learning objectives

- extract JSON from messy model output (prose, markdown fences)
- validate against a Pydantic schema
- retry with error feedback on validation failure
- understand why "please reply in JSON" is not reliable
- reuse tool schemas as output contracts

## 1. The problem: models don't output clean JSON

Typical raw model responses:

In [1]:
import re
import json

raw_outputs = [
    # Prose wrapper
    'Here is the invoice data:\n{\n  "invoice_number": "INV-001",\n  "total": 1234.56\n}',
    
    # Markdown fences
    '```json\n{"invoice_number": "INV-002", "total": 99.00}\n```',
    
    # Trailing comma (invalid JSON)
    '{"invoice_number": "INV-003", "total": 45.0,}',
    
    # String instead of number
    '{"invoice_number": "INV-004", "total": "1234.56"}',
    
    # Clean JSON
    '{"invoice_number": "INV-005", "total": 75.0}'
]

for i, out in enumerate(raw_outputs):
    print(f"=== Output {i+1} ===")
    print(out[:80])
    print()

=== Output 1 ===
Here is the invoice data:
{
  "invoice_number": "INV-001",
  "total": 1234.56
}

=== Output 2 ===
```json
{"invoice_number": "INV-002", "total": 99.00}
```

=== Output 3 ===
{"invoice_number": "INV-003", "total": 45.0,}

=== Output 4 ===
{"invoice_number": "INV-004", "total": "1234.56"}

=== Output 5 ===
{"invoice_number": "INV-005", "total": 75.0}



## 2. Robust JSON extraction

Strip markdown, find the first `{...}` block, attempt parse.

In [2]:
def extract_json(text: str) -> dict | None:
    """Extract first JSON object from text. Returns None if no valid JSON found."""
    # Remove markdown code fences
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    
    # Find first { and last }
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    
    candidate = text[start:end+1]
    try:
        parsed = json.loads(candidate)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        return None

# Test extraction
for i, out in enumerate(raw_outputs):
    result = extract_json(out)
    print(f"Output {i+1}: {result}")

Output 1: {'invoice_number': 'INV-001', 'total': 1234.56}
Output 2: {'invoice_number': 'INV-002', 'total': 99.0}
Output 3: None
Output 4: {'invoice_number': 'INV-004', 'total': '1234.56'}
Output 5: {'invoice_number': 'INV-005', 'total': 75.0}


## 3. Define the output schema with Pydantic

In [3]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

class Invoice(BaseModel):
    invoice_number: str
    date: Optional[str] = None
    total: float = Field(gt=0)
    vendor: Optional[str] = None

# Special case: not an invoice
class NotAnInvoice(BaseModel):
    error: str = "not_an_invoice"

# Test validation
test_cases = [
    {"invoice_number": "INV-001", "total": 1234.56},
    {"invoice_number": "INV-002", "total": "99.00"},  # string total
    {"invoice_number": "INV-003", "total": -10},       # negative total
    {"total": 50.0},                                       # missing invoice_number
    {"error": "not_an_invoice"},
]

for case in test_cases:
    if "error" in case:
        try:
            result = NotAnInvoice.model_validate(case)
            print(f"OK: {result}")
        except ValidationError as e:
            print(f"FAIL: {e.errors()[0]['msg']}")
    else:
        try:
            result = Invoice.model_validate(case)
            print(f"OK: {result}")
        except ValidationError as e:
            print(f"FAIL: {e.errors()[0]['msg']}")

OK: invoice_number='INV-001' date=None total=1234.56 vendor=None
OK: invoice_number='INV-002' date=None total=99.0 vendor=None
FAIL: Input should be greater than 0
FAIL: Field required
OK: error='not_an_invoice'


## 4. Format validation errors for retry feedback

In [4]:
def format_validation_errors(e: ValidationError) -> str:
    """Convert Pydantic ValidationError to human-readable feedback for the model."""
    lines = []
    for error in e.errors():
        field = ".".join(str(x) for x in error['loc'])
        msg = error['msg']
        received = error.get('input')
        if received is not None:
            lines.append(f"Field '{field}': {msg} (received: {json.dumps(received)}) ")
        else:
            lines.append(f"Field '{field}': {msg}")
    return "\n".join(lines)

# Test error formatting
try:
    Invoice.model_validate({"invoice_number": "INV-001", "total": "1234.56"})
except ValidationError as e:
    print(format_validation_errors(e))

## 5. The parse-and-retry loop (simulated)

We simulate the API call with a function that returns progressively better responses.

In [5]:
# Simulated API responses for testing the retry loop
SIMULATED_RESPONSES = [
    # Attempt 1: prose wrapper, string total
    'Here is the extracted invoice:\n```json\n{"invoice_number": "INV-001", "total": "1234.56"}\n```',
    
    # Attempt 2: clean JSON, missing vendor
    '{"invoice_number": "INV-001", "total": 1234.56}',
    
    # Attempt 3: perfect
    '{"invoice_number": "INV-001", "date": "2024-01-15", "total": 1234.56, "vendor": "Acme Corp"}'
]

call_count = 0

def mock_api_call(messages: list[dict], **kwargs) -> dict:
    """Simulates an API that improves on retry."""
    global call_count
    response = SIMULATED_RESPONSES[min(call_count, len(SIMULATED_RESPONSES)-1)]
    call_count += 1
    return {"choices": [{"message": {"content": response}}]}

def extract_content(response: dict) -> str:
    return response["choices"][0]["message"]["content"]

In [6]:
def structured_call_with_retry(
    model_class: type[BaseModel],
    messages: list[dict],
    max_retries: int = 3,
    api_call_fn=mock_api_call,
) -> BaseModel:
    """Parse-and-retry loop with error feedback."""
    schema_json = model_class.model_json_schema()

    system_prompt = f"""You are a structured data extraction engine.
Output ONLY valid JSON matching this exact schema:
{json.dumps(schema_json, indent=2)}

Rules:
- No prose, no markdown, no commentary
- All required fields must be present
- Types must match exactly (number not string)"""

    last_error = None
    full_messages = [{"role": "system", "content": system_prompt}, *messages]

    for attempt in range(max_retries):
        # Build messages for this attempt
        if attempt == 0:
            current_messages = full_messages
        else:
            current_messages = full_messages + [
                {
                    "role": "user",
                    "content": (
                        f"Previous response failed validation:\n{last_error}\n\n"
                        "Return corrected JSON only."
                    ),
                }
            ]

        # Call API
        response = api_call_fn(current_messages)
        raw = extract_content(response)

        # Extract JSON
        parsed = extract_json(raw)
        if parsed is None:
            last_error = f"No valid JSON found in response. Raw: {raw[:200]}"
            print(f"Attempt {attempt+1}: JSON extraction failed")
            continue

        # Validate
        try:
            validated = model_class.model_validate(parsed)
            print(f"Attempt {attempt+1}: SUCCESS")
            return validated
        except ValidationError as exc:
            last_error = format_validation_errors(exc)
            print(f"Attempt {attempt+1}: Validation failed - {exc.errors()[0]['msg']}")
            continue

    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {last_error}")


# Run the retry loop
call_count = 0
messages = [
    {
        "role": "user",
        "content": (
            "Invoice #INV-001\nDate: 2024-01-15\nVendor: Acme Corp\nTotal: $1,234.56"
        ),
    }
]

result = structured_call_with_retry(Invoice, messages, max_retries=3)
print(f"\nFinal result: {result}")
print(f"API calls made: {call_count}")


Attempt 1: SUCCESS

Final result: invoice_number='INV-001' date=None total=1234.56 vendor=None
API calls made: 1


## 6. Tool schemas as output contracts

The same schema used for function calling doubles as the output contract.

In [7]:
# Tool schema for extract_invoice
invoice_tool = {
    "type": "function",
    "function": {
        "name": "extract_invoice",
        "description": "Extract invoice fields from text",
        "parameters": Invoice.model_json_schema()
    }
}

import json
print(json.dumps(invoice_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "extract_invoice",
    "description": "Extract invoice fields from text",
    "parameters": {
      "properties": {
        "invoice_number": {
          "title": "Invoice Number",
          "type": "string"
        },
        "date": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "title": "Date"
        },
        "total": {
          "exclusiveMinimum": 0,
          "title": "Total",
          "type": "number"
        },
        "vendor": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "title": "Vendor"
        }
      },
      "required": [
        "invoice_number",
        "total"
      ],
      "title": "Invoice",
      "type": "object"
    }
  }
}


## 7. JSON mode vs structured output (conceptual)

| Feature | JSON Mode | Structured Output (strict) |
|---------|-----------|----------------------------|
| Syntax valid | Yes | Yes |
| Schema valid | No | Yes |
| Extra fields allowed | Yes | No |
| Type coercion | No | Yes (constrained) |
| Provider support | Most | OpenAI (latest), Anthropic (tools) |

JSON mode only guarantees valid JSON syntax. Structured output guarantees the schema.

## 8. Completion exercises

Do these by hand. No autocomplete.

In [8]:
# EXERCISE 1: Add a `model_validator` to Invoice that ensures
# if vendor is present, it's not empty string

# EXERCISE 2: Modify the retry loop to also handle the
# `NotAnInvoice` case - if the model returns {"error": "not_an_invoice"},
# return that instead of retrying

# EXERCISE 3: Add a max_tokens budget tracker that stops retrying
# if total tokens would exceed a limit

# EXERCISE 4: Write a test that simulates a model that alternates
# between two different invalid responses (never converges)

print("Exercises ready — implement by hand.")

Exercises ready — implement by hand.
